# 01 - Dataset Audit: RAVDESS

**Purpose:** Before writing any preprocessing or model code, we inspect what we actually have:
- How many files exist per actor
- What the filename encoding tells us (emotion, intensity, actor, statement, repetition)
- File formats and basic properties (video resolution, audio sample rate, duration)
- Class distribution (are emotions balanced?)

This notebook only reads and reports - it does not modify or copy any files.

In [ ]:
import os
from pathlib import Path
import pandas as pd

# Path to raw RAVDESS data
RAVDESS_ROOT = Path("../data/raw/ravdess")

print("Looking for data at:", RAVDESS_ROOT.resolve())
print("Exists:", RAVDESS_ROOT.exists())

: 

## Step 1: List all actor folders found

We expect folders named `Actor_01`, `Actor_02`, etc. Right now you may only have 1-3 actors downloaded - that's fine, this notebook works with however many you have.

In [ ]:
actor_folders = sorted([f for f in RAVDESS_ROOT.iterdir() if f.is_dir() and f.name.startswith("Actor_")])
print(f"Found {len(actor_folders)} actor folder(s):")
for f in actor_folders:
    print(" -", f.name)

## Step 2: Decode the RAVDESS filename convention

Each file is named like `03-01-06-01-02-01-12.mp4` - seven numbers separated by dashes.

According to the official RAVDESS documentation, each position means:

| Position | Meaning | Values |
|---|---|---|
| 1 | Modality | 01=full-AV, 02=video-only, 03=audio-only |
| 2 | Vocal channel | 01=speech, 02=song |
| 3 | Emotion | 01=neutral, 02=calm, 03=happy, 04=sad, 05=angry, 06=fearful, 07=disgust, 08=surprised |
| 4 | Emotional intensity | 01=normal, 02=strong (note: no strong intensity for neutral) |
| 5 | Statement | 01="Kids are talking by the door", 02="Dogs are sitting by the door" |
| 6 | Repetition | 01=1st repetition, 02=2nd repetition |
| 7 | Actor | 01 to 24 (odd=male, even=female) |

We'll write a function to parse this directly from filenames - this becomes the foundation of our label-mapping work in Notebook 02.

In [ ]:
EMOTION_MAP = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised",
}

MODALITY_MAP = {"01": "full_AV", "02": "video_only", "03": "audio_only"}
CHANNEL_MAP = {"01": "speech", "02": "song"}
INTENSITY_MAP = {"01": "normal", "02": "strong"}

def parse_ravdess_filename(filename: str) -> dict:
    """Parse a RAVDESS filename into its component labels.
    Example input: '01-01-06-01-02-01-12.mp4'
    """
    stem = Path(filename).stem
    parts = stem.split("-")
    if len(parts) != 7:
        return None
    modality, channel, emotion, intensity, statement, repetition, actor = parts
    return {
        "filename": filename,
        "modality": MODALITY_MAP.get(modality, modality),
        "channel": CHANNEL_MAP.get(channel, channel),
        "emotion": EMOTION_MAP.get(emotion, emotion),
        "intensity": INTENSITY_MAP.get(intensity, intensity),
        "statement": statement,
        "repetition": repetition,
        "actor_id": int(actor),
        "gender": "male" if int(actor) % 2 == 1 else "female",
    }

test_result = parse_ravdess_filename("02-01-06-01-02-01-12.mp4")
test_result

## Step 3: Walk every actor folder and parse every file

This builds a full table (DataFrame) of everything we have - one row per file, with all labels decoded.

In [ ]:
all_records = []

for actor_folder in actor_folders:
    for file in actor_folder.iterdir():
        if file.suffix.lower() in [".mp4", ".wav"]:
            parsed = parse_ravdess_filename(file.name)
            if parsed:
                parsed["filepath"] = str(file)
                parsed["file_size_kb"] = round(file.stat().st_size / 1024, 1)
                all_records.append(parsed)

df = pd.DataFrame(all_records)
print(f"Total files parsed: {len(df)}")
df.head(10)

## Step 4: Basic checks

- How many files per actor?
- How many per modality (full_AV vs video_only vs audio_only)?
- Emotion class distribution

These checks matter because: if modality counts look wrong, our download/extraction may be incomplete. If emotion classes are wildly imbalanced, we'll need class weighting later (as planned in Phase 0).

In [ ]:
print("Files per actor:")
print(df.groupby("actor_id").size())
print()
print("Files per modality:")
print(df["modality"].value_counts())
print()
print("Files per channel (speech vs song):")
print(df["channel"].value_counts())

In [ ]:
df_speech_av = df[(df["channel"] == "speech") & (df["modality"] == "full_AV")]

print(f"Full audio-video speech files: {len(df_speech_av)}")
print()
print("Emotion distribution (speech, full AV only):")
print(df_speech_av["emotion"].value_counts())

## Step 5: Save the parsed metadata table

We save this as a CSV in `data/interim/` - this becomes the foundation for Notebook 02 (label mapping) and later for building our train/val/test splits.

In [ ]:
output_path = Path("../data/interim/ravdess_file_index.csv")
df.to_csv(output_path, index=False)
print(f"Saved {len(df)} rows to {output_path.resolve()}")

## Next steps

- Once all 24 actors are downloaded, re-run this notebook top-to-bottom.
- This index CSV feeds directly into Notebook 02 (label mapping).
- We have NOT touched audio extraction, video frame extraction, or face detection yet.